In [42]:
# Работа с данными
import pandas as pd
import numpy as np
import datetime
import itertools
# import category_encoders as ce

# Общие функции
import sys
sys.path.append("/home/jovyan/share/1libs/Functions/") # go to parent dir
from oskar_ml import *

# Визуализация
import matplotlib.pyplot as plt
from copy import deepcopy
import seaborn as sns
from scipy import stats


# Игнорирование системных предупреждений
import warnings
warnings.filterwarnings('ignore')

# Манипуляция данными
import gc #сбор мусора
gc.enable()

# Визуализация процесса расчёта
from tqdm import tqdm, tqdm_notebook
tqdm.pandas()

# Отобразить все колонки
from IPython.display import display
pd.set_option('display.max_columns', None)
# Отобразить 230 строк
pd.set_option("display.max_rows", 230)

# Модели
from catboost import CatBoostRegressor, Pool, EShapCalcType, EFeaturesSelectionAlgorithm, sum_models
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Разделение данных
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.model_selection import StratifiedKFold,  RandomizedSearchCV

# Метрики
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn import metrics

# Оптимизация гиперпараметров
import optuna

In [43]:
df = pd.read_parquet('/home/jovyan/old_home/Litigant/data/processed/df_final_2.parquet')
df.shape

(103847, 380)

In [44]:
df = df.rename({'EMRValue': 'EMR_VALUE'}, axis=1)

In [45]:
ftrs = [
    'INCIDENT_NUMBER',
    'LOSS_NUMBER',
    'FILIAL',
    'EVENT_CREATED_BY_GIBDD_FLAG',
    'VICTIM_MAX_WEIGHT',
    'RECIEVE_METHOD',
    'APPLICANT_FORM',
    'APPLICANT_AGE',
    'VICTIM_VEHICLE_CATEGORY',
    'GUILTY_CAPACITY_ENGINE',
    'VICTIM_VEHICLE_AGE',
    'EVENT_DATE',
    'PAYMENT_ORDER_DATE_TIME',
    'AMOUNT_REPAIR',
    'LOSS_UNIT_ZONE',
    'VICTIM_VEHICLE_COUNTRY']

In [46]:
df_test = df[ftrs][1:2]

In [47]:
for col in [col for col in df_test.columns if 'DATE' in col]:
    df_test[col] = df_test[col].astype(str)

In [48]:
import json
import requests

header = {'Content-Type': 'application/json;charset=utf-8', \
                  'Accept': 'application/json'}

In [49]:
import requests
import json
import numpy as np

# 1. Заменяем NaN и inf на None (в JSON это станет null)
df_clean = df_test.replace({np.nan: None, np.inf: None, -np.inf: None})

# 2. ВАЖНО: Приводим все колонки с датами к строковому формату ISO 8601 (как в вашем примере)
# Pandas по умолчанию может отправлять даты с микросекундами, что ломает сервер
date_cols = df_clean.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns
for col in date_cols:
    df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

# 3. Преобразуем DataFrame в список словарей
records = df_clean.to_dict(orient='records')

# 4. ОТЛАДКА: Выводим первую запись, чтобы визуально сравнить с эталоном
print("--- ПЕРВАЯ ЗАПИСЬ, КОТОРАЯ УЙДЕТ НА СЕРВЕР ---")
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print("------------------------------------------------\n")

# TEST
url = "http://perf-pydsp-test1.vsk.ru:10418/predict/querulus"
# PROD
# url = "http://perf-pydsp-cls.vsk.ru:10318/predict/querulus"
# headers = {"Content-Type": "application/json"}

# ==========================================
# ВАРИАНТ А: С оберткой (как в первом запросе)
# ==========================================
data_to_send = {
    'main_model': 'querulus_ansamble_2026_04_18_v1',
    'main_request': records
}

# print("Отправляем Вариант А (с оберткой main_model)...")
# response = requests.post(url, headers=headers, json=data_to_send)
# print(f"Статус: {response.status_code}")
# print(f"Ответ: {response.text}\n")

# ==========================================
# ВАРИАНТ Б: БЕЗ обертки (просто массив)
# ==========================================
# Раскомментируйте этот блок, если Вариант А снова вернет 500.
# Очень часто API ожидает именно просто массив объектов, без main_model.

print("Отправляем Вариант Б (просто массив)...")
response_b = requests.post(url, headers=headers, json=records)
print(f"Статус: {response_b.status_code}")
print(f"Ответ: {response_b.text}")

--- ПЕРВАЯ ЗАПИСЬ, КОТОРАЯ УЙДЕТ НА СЕРВЕР ---
{
  "INCIDENT_NUMBER": 6183770.0,
  "LOSS_NUMBER": 9135252.0,
  "FILIAL": "ВСК-Москва",
  "EVENT_CREATED_BY_GIBDD_FLAG": 1,
  "VICTIM_MAX_WEIGHT": null,
  "RECIEVE_METHOD": "Лично",
  "APPLICANT_FORM": "Потерпевший",
  "APPLICANT_AGE": 57.0,
  "VICTIM_VEHICLE_CATEGORY": "Средний 7+",
  "GUILTY_CAPACITY_ENGINE": null,
  "VICTIM_VEHICLE_AGE": 10.0,
  "EVENT_DATE": "2018-06-01 16:35:00",
  "PAYMENT_ORDER_DATE_TIME": "2023-03-05 21:00:00",
  "AMOUNT_REPAIR": 128775.0,
  "LOSS_UNIT_ZONE": "Зона Московского региона",
  "VICTIM_VEHICLE_COUNTRY": "Япония"
}
------------------------------------------------

Отправляем Вариант Б (просто массив)...
Статус: 200
Ответ: {
    "version": "1.2.0",
    "threshold": 0.6,
    "CPMComment": "shap_clf: p=0.5267 | VICTIM_MAX_WEIGHT=+48.19%, VICTIM_VEHICLE_AGE=-27.57%, GUILTY_CAPACITY_ENGINE=+20.32%, EVENT_CREATED_BY_GIBDD_FLAG=-11.19%, APPLICANT_FORM=-7.35% | shap_reg: VICTIM_VEHICLE_COUNTRY=+10304.2660, LOSS_U

In [50]:
import requests
import json
import numpy as np
import pandas as pd
from datetime import datetime

# ==========================================
# ДОБАВЛЕНИЕ ТЕСТОВОЙ ЗАПИСИ АВТОЮРИСТА
# ==========================================

# Создаем запись автоюриста с ФИО и датой рождения
# lawyer_record = {
#     'APPLICANT': 'Иванов Иван Иванович',
#     'APPLICANT_BIRTHDAY': datetime(1980, 5, 15),  # или строка '1980-05-15'
#     # Добавьте остальные поля, которые есть в вашем df_test
# }

# Добавляем запись в df_test
# Вариант 1: Если df_test уже существует
df_test['APPLICANT'] = 'Бобушкин Сергей Алексеевич'
df_test['APPLICANT_BIRTHDAY'] = datetime(1976, 11, 19)

# Вариант 2: Если нужно добавить несколько записей
# lawyer_records = [
#     {'fio': 'Иванов Иван Иванович', 'birth_date': datetime(1980, 5, 15)},
#     {'fio': 'Петров Петр Петрович', 'birth_date': datetime(1975, 8, 20)},
# ]
# df_test = pd.concat([df_test, pd.DataFrame(lawyer_records)], ignore_index=True)

print(f"Добавлена запись автоюриста. Всего записей: {len(df_test)}")

Добавлена запись автоюриста. Всего записей: 1


In [51]:
# ==========================================
# ОСНОВНАЯ ОБРАБОТКА (ваш код)
# ==========================================

# 1. Заменяем NaN и inf на None (в JSON это станет null)
df_clean = df_test.replace({np.nan: None, np.inf: None, -np.inf: None})

# 2. ВАЖНО: Приводим все колонки с датами к строковому формату ISO 8601
date_cols = df_clean.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns
for col in date_cols:
    df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

# 3. Преобразуем DataFrame в список словарей
records = df_clean.to_dict(orient='records')

# 4. ОТЛАДКА: Выводим ПОСЛЕДНЮЮ запись (добавленного автоюриста)
print("\n--- ПОСЛЕДНЯЯ ЗАПИСЬ (АВТОЮРИСТ), КОТОРАЯ УЙДЕТ НА СЕРВЕР ---")
print(json.dumps(records[-1], ensure_ascii=False, indent=2))
print("------------------------------------------------\n")

# 5. ОТЛАДКА: Выводим первую запись для сравнения
print("--- ПЕРВАЯ ЗАПИСЬ ДЛЯ СРАВНЕНИЯ ---")
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print("------------------------------------------------\n")



--- ПОСЛЕДНЯЯ ЗАПИСЬ (АВТОЮРИСТ), КОТОРАЯ УЙДЕТ НА СЕРВЕР ---
{
  "INCIDENT_NUMBER": 6183770.0,
  "LOSS_NUMBER": 9135252.0,
  "FILIAL": "ВСК-Москва",
  "EVENT_CREATED_BY_GIBDD_FLAG": 1,
  "VICTIM_MAX_WEIGHT": null,
  "RECIEVE_METHOD": "Лично",
  "APPLICANT_FORM": "Потерпевший",
  "APPLICANT_AGE": 57.0,
  "VICTIM_VEHICLE_CATEGORY": "Средний 7+",
  "GUILTY_CAPACITY_ENGINE": null,
  "VICTIM_VEHICLE_AGE": 10.0,
  "EVENT_DATE": "2018-06-01 16:35:00",
  "PAYMENT_ORDER_DATE_TIME": "2023-03-05 21:00:00",
  "AMOUNT_REPAIR": 128775.0,
  "LOSS_UNIT_ZONE": "Зона Московского региона",
  "VICTIM_VEHICLE_COUNTRY": "Япония",
  "APPLICANT": "Бобушкин Сергей Алексеевич",
  "APPLICANT_BIRTHDAY": "1976-11-19T00:00:00"
}
------------------------------------------------

--- ПЕРВАЯ ЗАПИСЬ ДЛЯ СРАВНЕНИЯ ---
{
  "INCIDENT_NUMBER": 6183770.0,
  "LOSS_NUMBER": 9135252.0,
  "FILIAL": "ВСК-Москва",
  "EVENT_CREATED_BY_GIBDD_FLAG": 1,
  "VICTIM_MAX_WEIGHT": null,
  "RECIEVE_METHOD": "Лично",
  "APPLICANT_FORM": "

In [52]:
# ==========================================
# ОТПРАВКА НА СЕРВЕР
# ==========================================

# TEST
# url = "http://perf-pydsp-test1.vsk.ru:10418/predict/querulus"
# PROS
url = "http://perf-pydsp-cls.vsk.ru:10318/predict/querulus"
headers = {"Content-Type": "application/json"}

# ==========================================
# ВАРИАНТ А: С оберткой
# ==========================================
data_to_send = {
    'main_model': 'querulus_ansamble_2026_22_04_v1',
    'main_request': records
}

# print("Отправляем Вариант А (с оберткой main_model)...")
# response = requests.post(url, headers=headers, json=data_to_send)
# print(f"Статус: {response.status_code}")
# print(f"Ответ: {response.text}\n")

# ==========================================
# ВАРИАНТ Б: БЕЗ обертки (на случай если А не сработает)
# ==========================================
print("Отправляем Вариант Б (просто массив)...")
response_b = requests.post(url, headers=headers, json=records)
print(f"Статус: {response_b.status_code}")
print(f"Ответ: {response_b.text}")

Отправляем Вариант Б (просто массив)...
Статус: 200
Ответ: {
    "version": "1.2.0",
    "threshold": 0.6,
    "CPMComment": "shap_clf: p=0.5267 | VICTIM_VEHICLE_AGE=+48.19%, APPLICANT_FORM=-27.57%, RECIEVE_METHOD=+20.32%, FILIAL=-11.19%, VICTIM_VEHICLE_CATEGORY=-7.35% | shap_reg: APPLY_DELAY=+10304.2660, LOSS_UNIT_ZONE=+9314.0760, AMOUNT_REPAIR=+8990.4389, VICTIM_VEHICLE_COUNTRY=-886.3854",
    "classification_proba": [
        0.53
    ],
    "classification_predictions": [
        0.0
    ],
    "regression_predictions": [
        90799.95
    ],
    "predictions": "[90799.95]",
    "comment": "2408891"
}


In [9]:
%%time

import pandas as pd
import numpy as np
import requests
import json
import re
from tqdm import tqdm
from time import sleep

# ============================================================
# 1. КОНСТАНТЫ (ЗАДАЙТЕ СВОИ ПУТИ)
# ============================================================
DATA_PATH = '/home/jovyan/old_home/Litigant/data/processed/df_final_2.parquet'
CONTACTS_PATH = '/home/jovyan/old_home/Litigant/lawyer_service/СВОД_preprocessed.xlsx'   # <-- ЗАДАЙТЕ ПУТЬ К ФАЙЛУ С КОНТАКТАМИ
OUTPUT_PATH  = '/home/jovyan/old_home/Litigant/lawyer_service/lawyer_servis_test_results.xlsx'     # <-- ЗАДАЙТЕ ПУТЬ ДЛЯ РЕЗУЛЬТАТА
URL = "http://perf-pydsp-test1.vsk.ru:10418/predict/querulus"

# Настройки сети и задержек
MAX_RETRIES = 3
RETRY_DELAY = 5       # секунд между попытками при ошибке соединения
REQUEST_DELAY = 5    # секунд между запросами (и успешными, и неуспешными)
MAX_CONSECUTIVE_ERRORS = 5  # Прерываем скрипт, если 5 запросов подряд упали с ошибкой

# ============================================================
# 2. ЗАГРУЗКА И ПОДГОТОВКА ОСНОВНЫХ ДАННЫХ
# ============================================================
df = pd.read_parquet(DATA_PATH)
df = df.rename({'EMRValue': 'EMR_VALUE'}, axis=1)

ftrs = [
    'INCIDENT_NUMBER', 'LOSS_NUMBER', 'FILIAL', 'EVENT_CREATED_BY_GIBDD_FLAG',
    'VICTIM_MAX_WEIGHT', 'RECIEVE_METHOD', 'APPLICANT_FORM', 'APPLICANT_AGE',
    'VICTIM_VEHICLE_CATEGORY', 'GUILTY_CAPACITY_ENGINE', 'VICTIM_VEHICLE_AGE',
    'EVENT_DATE', 'PAYMENT_ORDER_DATE_TIME', 'AMOUNT_REPAIR', 'LOSS_UNIT_ZONE',
    'VICTIM_VEHICLE_COUNTRY'
]

df_test = df[ftrs][1:2]
df_test['FILIAL'] = 'Марийский'

for col in [c for c in df_test.columns if 'DATE' in c]:
    df_test[col] = df_test[col].astype(str)

df_clean = df_test.replace({np.nan: None, np.inf: None, -np.inf: None})

date_cols = df_clean.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns
for col in date_cols:
    df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

base_record = df_clean.to_dict(orient='records')[0]

# ============================================================
# 3. ЗАГРУЗКА КОНТАКТНЫХ ДАННЫХ
# ============================================================
contacts_df = pd.read_excel(CONTACTS_PATH)

col_mapping = {
    'ФИО': 'ФИО', 'ДАТА РОЖДЕНИЯ': 'ДАТА_РОЖДЕНИЯ', 'ИНН': 'ИНН',
    'ТЕЛЕФОН': 'ТЕЛЕФОН', 'E-MAIL': 'EMAIL', 'АДРЕС': 'АДРЕС',
}
contacts_df = contacts_df.rename(columns={k: v for k, v in col_mapping.items() if k in contacts_df.columns})

if 'ДАТА_РОЖДЕНИЯ' in contacts_df.columns:
    contacts_df['ДАТА_РОЖДЕНИЯ'] = pd.to_datetime(
        contacts_df['ДАТА_РОЖДЕНИЯ'], errors='coerce'
    ).dt.strftime('%Y-%m-%d')

# ============================================================
# 4. ОПРЕДЕЛЕНИЕ ТИПОВ ЗАПРОСОВ
# ============================================================
CONTACT_TYPES = [
    ('APPLICANT_ФИО+ДР',   [('APPLICANT', 'ФИО'), ('APPLICANT_BIRTHDAY', 'ДАТА_РОЖДЕНИЯ')]),
    ('APPLICANT_ИНН',      [('APPLICANT_INN', 'ИНН')]),
    ('APPLICANT_ТЕЛЕФОН',  [('APPLICANT_PHONE', 'ТЕЛЕФОН')]),
    ('APPLICANT_EMAIL',    [('APPLICANT_EMAIL', 'EMAIL')]),
    ('APPLICANT_АДРЕС',    [('APPLICANT_ADDRESS', 'АДРЕС')]),
    ('PAYMENT_RECEPIENT_ФИО+ДР',   [('PAYMENT_RECEPIENT', 'ФИО'), ('PAYMENT_RECEPIENT_BIRTHDAY', 'ДАТА_РОЖДЕНИЯ')]),
    ('PAYMENT_RECEPIENT_ИНН',      [('PAYMENT_RECEPIENT_INN', 'ИНН')]),
    ('PAYMENT_RECEPIENT_ТЕЛЕФОН',  [('PAYMENT_RECEPIENT_PHONE', 'ТЕЛЕФОН')]),
    ('PAYMENT_RECEPIENT_EMAIL',    [('PAYMENT_RECEPIENT_EMAIL', 'EMAIL')]),
    ('PAYMENT_RECEPIENT_АДРЕС',    [('PAYMENT_RECEPIENT_ADDRESS', 'АДРЕС')]),
    ('PHONE_FOR_RECEIPT',  [('PHONE_FOR_SENDING_THE_RECEIPT', 'ТЕЛЕФОН')]),
    ('EMAIL_FOR_RECEIPT',  [('EMAIL_FOR_SENDING_THE_RECEIPT', 'EMAIL')]),
]

# ============================================================
# 5. ФУНКЦИЯ ОТПРАВКИ ЗАПРОСА С RETRY
# ============================================================
def send_prediction_request(base_record, extra_fields, max_retries=MAX_RETRIES):
    payload = [base_record.copy()]
    for k, v in extra_fields.items():
        payload[0][k] = v

    last_error = None
    for attempt in range(max_retries):
        try:
            return requests.post(URL, headers={"Content-Type": "application/json"}, json=payload, timeout=60)
        except requests.exceptions.ConnectionError as e:
            last_error = f"Connection error: {e}"
            if attempt < max_retries - 1: sleep(RETRY_DELAY)
        except requests.exceptions.Timeout as e:
            last_error = f"Timeout error: {e}"
            if attempt < max_retries - 1: sleep(RETRY_DELAY)
        except Exception as e:
            last_error = f"Unexpected error: {e}"
            break
    
    class FakeResp:
        status_code = 0
        text = last_error or "Unknown error"
        def json(self): raise ValueError(last_error)
    return FakeResp()

# ============================================================
# 6. ФУНКЦИЯ ПАРСИНГА PREDICTIONS
# ============================================================
def parse_predictions(predictions_str):
    if not predictions_str or predictions_str in ['SKIPPED', 'ERROR']: return None
    try:
        match = re.search(r'\[([0-9.-]+)\]', str(predictions_str))
        if match: return float(match.group(1))
    except: pass
    return None

# ============================================================
# 7. ПРОВЕРКА ДОСТУПНОСТИ СЕРВЕРА
# ============================================================
print("Проверка доступности сервера...")
test_resp = send_prediction_request(base_record, {}, max_retries=1)
if test_resp.status_code == 0:
    raise ConnectionError(f"❌ Сервер недоступен! Детали: {test_resp.text}")
else:
    print(f"✅ Сервер доступен (статус: {test_resp.status_code})")
    sleep(REQUEST_DELAY)

# ============================================================
# 8. КОНТРОЛЬНЫЙ ЗАПРОС БЕЗ КОНТАКТОВ
# ============================================================
results = []
print("\nОтправка контрольного запроса БЕЗ_КОНТАКТОВ...")
resp = send_prediction_request(base_record, {})
control_result = {'predictions': None, 'contact_type': 'БЕЗ_КОНТАКТОВ', 'row_index': -1, 'http_status': resp.status_code}

if resp.status_code == 200:
    data = resp.json()
    control_result['predictions'] = parse_predictions(data.get('predictions'))
    print(f"✅ Контрольный запрос: predictions = {control_result['predictions']}")
else:
    control_result['error'] = resp.text[:500]
    print(f"❌ Контрольный запрос вернул статус {resp.status_code}")

results.append(control_result)
sleep(REQUEST_DELAY)

# ============================================================
# 9. ИТЕРАТИВНАЯ ОТПРАВКА ЗАПРОСОВ С ЛОГИРОВАНИЕМ И ПРЕДОХРАНИТЕЛЕМ
# ============================================================
abort_run = False
consecutive_errors = 0

for idx, row in tqdm(enumerate(contacts_df[:len(contacts_df) // 2].itertuples()), total=len(contacts_df), desc="Обработка контактов"):
    if abort_run:
        break
        
    for contact_type, fields in CONTACT_TYPES:
        extra_fields = {}
        valid_fields_count = 0

        for api_name, col_name in fields:
            val = getattr(row, col_name, None)
            if pd.notna(val) and val is not None and str(val).strip() != '':
                extra_fields[api_name] = val
                valid_fields_count += 1

        result_row = {
            'predictions': None, 'contact_type': contact_type,
            'row_index': idx, 'http_status': None,
        }
        for api_name, val in extra_fields.items():
            result_row[f'sent_{api_name}'] = val

        if valid_fields_count == 0:
            result_row['http_status'] = 'SKIPPED'
            results.append(result_row)
            continue

        # ОТПРАВКА ЗАПРОСА
        resp = send_prediction_request(base_record, extra_fields)
        result_row['http_status'] = resp.status_code

# ПРОВЕРКА НА ОШИБКИ (ПРЕДОХРАНИТЕЛЬ) 
        if resp.status_code != 200:
            consecutive_errors += 1
            err_text = resp.text[:80].replace('\n', ' ') if hasattr(resp, 'text') and resp.text else "Empty"
            tqdm.write(f"❌ [{idx}] {contact_type}: HTTP {resp.status_code} | {err_text}")
            
            result_row['error'] = resp.text[:500] if hasattr(resp, 'text') else "Unknown"
            results.append(result_row)
            
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                tqdm.write(f"\n🚨 ПРЕДОХРАНИТЕЛЬ СРАБОТАЛ: {MAX_CONSECUTIVE_ERRORS} ошибок подряд. Сервер лежит. Прерываем прогон!")
                abort_run = True
                sleep(REQUEST_DELAY)
                break
        else:
            # УСПЕХ: сбрасываем счетчик ошибок, парсим ответ
            consecutive_errors = 0
            try:
                data = resp.json()
                result_row['predictions'] = parse_predictions(data.get('predictions'))
                result_row['regression_predictions'] = data.get('regression_predictions', [None])[0]
                result_row['classification_predictions'] = data.get('classification_predictions', [None])[0]
                result_row['classification_proba'] = data.get('classification_proba', [None])[0]
                result_row['comment'] = data.get('comment')
                
                # Логируем результат с учетом -100
                pred_val = result_row['predictions']
                if pred_val == -100.0:
                    tqdm.write(f"⚠️ [{idx}] {contact_type}: pred = -100.0 (специальное значение)")
                else:
                    tqdm.write(f"✅ [{idx}] {contact_type}: pred={pred_val}")
            except Exception as e:
                result_row['error'] = f"JSON parse error: {e}"
                tqdm.write(f"⚠️ [{idx}] {contact_type}: Ошибка парсинга JSON")

            results.append(result_row)

        # Пауза после каждого запроса (и успешного, и неуспешного)
        sleep(REQUEST_DELAY)

# ============================================================
# 10. СОХРАНЕНИЕ И АНАЛИЗ РЕЗУЛЬТАТОВ
# ============================================================
results_df = pd.DataFrame(results)

cols_order = ['predictions', 'contact_type', 'row_index', 'http_status']
sent_cols = [c for c in results_df.columns if c.startswith('sent_')]
other_cols = [c for c in results_df.columns if c not in cols_order and c not in sent_cols]
results_df = results_df[cols_order + sent_cols + other_cols]

results_df.to_excel(OUTPUT_PATH, index=False)

# ============================================================
# 11. ФИНАЛЬНАЯ СТАТИСТИКА
# ============================================================
total = len(results_df)
processed = len(results_df[results_df['http_status'] == 200])
skipped = len(results_df[results_df['http_status'] == 'SKIPPED'])
errors = len(results_df[(results_df['http_status'] != 200) & (results_df['http_status'] != 'SKIPPED')])
pred_zero = len(results_df[results_df['predictions'] == 0.0])
pred_positive = len(results_df[results_df['predictions'] > 0.0])
pred_minus_100 = len(results_df[results_df['predictions'] == -100.0])  # Отслеживание -100

print(f"\n{'='*50}")
if abort_run:
    print("⚠️ ВНИМАНИЕ: Прогон был прерван досрочно из-за ошибок сервера!")
print(f"Всего строк в отчете: {total}")
print(f"Успешно (200 OK): {processed}")
print(f"Ошибок сервера: {errors}")
print(f"Пропущено (нет данных): {skipped}")
print(f"predictions == 0.0: {pred_zero}")
print(f"predictions > 0.0: {pred_positive}")
print(f"predictions == -100.0: {pred_minus_100}")  # Отдельная строка для -100
print(f"{'='*50}")

Проверка доступности сервера...
✅ Сервер доступен (статус: 200)

Отправка контрольного запроса БЕЗ_КОНТАКТОВ...
✅ Контрольный запрос: predictions = 90799.95


Обработка контактов:   0%|          | 0/217 [00:02<?, ?it/s]

✅ [0] APPLICANT_ФИО+ДР: pred=90799.95


Обработка контактов:   0%|          | 0/217 [00:09<?, ?it/s]

✅ [0] APPLICANT_ТЕЛЕФОН: pred=90799.95


Обработка контактов:   0%|          | 0/217 [00:16<?, ?it/s]

... (1,569 outputs left out)

Обработка контактов:  49%|████▉     | 107/217 [1:35:40<2:12:23, 72.21s/it]

✅ [107] EMAIL_FOR_RECEIPT: pred=90799.95


Обработка контактов:  50%|████▉     | 108/217 [1:35:45<1:36:38, 53.20s/it]



Всего строк в отчете: 1297
Успешно (200 OK): 789
Ошибок сервера: 0
Пропущено (нет данных): 508
predictions == 0.0: 0
predictions > 0.0: 789
predictions == -100.0: 0
CPU times: user 5.22 s, sys: 2.02 s, total: 7.24 s
Wall time: 1h 36min 2s
